In [1]:
import pandas as pd
import json

In [2]:
files = {
    "random": "random_metrics.json",
    "popular": "popular_metrics.json",
    "logreg": "logreg_metrics.json",
    "DSSM": "DSSM_metrics.json",
    "DSSM_with_features": "DSSM_with_features_metrics.json",
    "popular+DSSM": "union_PopularDSSM_metrics.json",
    "popular+logreg": "union_PopularLogreg_metrics.json",
    "popular+DSSM_with_features": "union_PopularDSSMwithFeatures_metrics.json",
    "DSSM+logreg": "union_DSSMLogreg_metrics.json",
    "DSSM_with_features+logreg": "union_DSSMwithFeaturesLogreg_metrics.json",
    "popular+logreg+DSSM_with_features": "union_PopularDSSMwithFeaturesLogreg_metrics.json",
}

In [3]:
metrics = dict()
for k,v in files.items():
    with open("result_metrics/" + v, "r", encoding="utf-8") as f:
        metrics[k] = json.load(f)

len(metrics)

11

In [4]:
metrics_df = pd.DataFrame(metrics)

metrics_df

,random,popular,logreg,DSSM,DSSM_with_features,popular+DSSM,popular+logreg,popular+DSSM_with_features,DSSM+logreg,DSSM_with_features+logreg,popular+logreg+DSSM_with_features
k,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000
users_to_validate,3868.000000,3868.000000,3868.000000,3868.000000,3868.000000,3868.000000,3868.000000,3868.000000,3868.000000,3868.000000,3868.000000
mapk,0.001641,0.017627,0.000421,0.007976,0.009434,0.017634,0.017075,0.017742,0.007512,0.008922,0.017431
hit_revenue_per_user,12.338547,18.953750,9.776073,21.736842,25.736578,24.271587,11.646657,27.286755,12.624930,15.640098,20.317420
total_hit_revenue,47725.500000,73313.105000,37813.850000,84078.105000,99549.085000,93882.500000,45049.270000,105545.170000,48833.230000,60495.900000,78587.780000
spend_coverage,0.042477,0.071227,0.034924,0.071711,0.084052,0.086440,0.044405,0.095610,0.042034,0.051204,0.071278
unique_items,2443.000000,200.000000,200.000000,2443.000000,2443.000000,2443.000000,200.000000,2440.000000,2443.000000,2440.000000,2430.000000
item_coverage_per_slot,0.003158,0.000259,0.000259,0.003158,0.003158,0.003158,0.000259,0.003154,0.003158,0.003154,0.003173
top_item_share,0.000496,0.005000,0.005000,0.001007,0.001157,0.005495,0.005000,0.005845,0.005314,0.005341,0.005779


### Легенда:

Мы - DS-команда маркетплейса и одна из наших бизнес-задач - построить новый кандидатогенератор. Уже на проде существует logreg - он был сделан другой командой (на самом деле мы сделали... ну это так, для бизнес-кейса) - нам нужно выкатить решение на АБ, в которое мы верим, что оно окажется лучше. 

Что значит "верим, что оно окажется лучше"? В компании принято использовать перед выкаткой на АБ фреймворк для валидации кандидатогенераторов - код, который берёт предикты разных моделей, подготовленных на основе истории (прошлое; дата Х), берёт **реальные** взаимодействия в будущем валидационном окне (дата Х; будущее) и считает метрики. Вот такая прокси-метрика к реальным ожидаемым результатам по АБ, в которую в компании верят. Никакие train/test/val split внутри исследовательских ноутбуков не считается за аргумент для выхода на АБ - доказательство нужно только side-to-side на этом валидационном фреймворке.

Дальше правило такое: есть текущее состояние прода. На нём работает такой-то ансамбль кандидатогенераторов (в нашем случае на проде - только logreg). Нам доступен такой АБ-тест: control - текущий ансамбль, test - текущий ансамбль + наша новая модель рекомендаций.

#### Компания даёт добро на этот АБ-тест, если мы показываем, что logreg+(наша новая модель рекомендаций) лучше, чем просто logreg, по метрике "total_hit_revenue".


"total_hit_revenue" считается так: например, сегодня `2015-06-01`. На ретроспективном запуске к дате `2015-05-01` наша модель подготовила для юзера 200 рекомендаций, на валидационном окне `2015-05-01` + 30 дней он реально купил из этих товаров только 10, и каждый из них стоит 10$. Тогда считаем ревенью по хитам (попаданиям) - 10*10 = 100. Так делаем для всех юзеров и суммируем.

### Самый выигрышный кандидат, удовлетворяющий условиям: DSSM_with_features+logreg

деплоим модель DSSM_with_features и отправляем на АБ!